In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import random
import numpy as np
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import haversine_distances
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from matplotlib.patches import PathPatch
from matplotlib.collections import PatchCollection, LineCollection
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
os.makedirs('outputs', exist_ok=True)

##### Load Data

In [ ]:
df = pd.read_csv('data/df_final.csv')
df['Year_Month'] = pd.to_datetime(df['Year_Month'])
df = df.sort_values(['Kab/Kota', 'Year_Month']).reset_index(drop=True)

print(f'Shape         : {df.shape}')
print(f'Rentang waktu : {df["Year_Month"].min()} -> {df["Year_Month"].max()}')
print(f'Kab/Kota      : {df["Kab/Kota"].nunique()}')
df.head()

##### Graph Construction - Adjacency Matrix

In [ ]:
coords = (
    df[['Kab/Kota', 'latitude', 'longitude']]
    .drop_duplicates()
    .sort_values('Kab/Kota')
    .reset_index(drop=True)
)

NODE_LIST = coords['Kab/Kota'].tolist()
N_NODES   = len(NODE_LIST)
node2idx  = {name: i for i, name in enumerate(NODE_LIST)}

print(f'Jumlah node (kab/kota): {N_NODES}')

In [ ]:
latlon_rad = np.radians(coords[['latitude', 'longitude']].values)
D_km = haversine_distances(latlon_rad) * 6371  

# Parameter 
K     = 5    
SIGMA = 25   

np.fill_diagonal(D_km, np.inf)  

# Adjacency matrix
A_hybrid = np.zeros((N_NODES, N_NODES), dtype=np.float32)
coba = []

for i in range(N_NODES):
    knn_idx = np.argsort(D_km[i])[:K]
    cur = []
    for j in knn_idx:
        cur.append(D_km[i,j])

        w = float(np.exp(-(D_km[i, j] ** 2) / (2 * SIGMA ** 2)))
        A_hybrid[i, j] = w
        A_hybrid[j, i] = w  
    coba.append(np.array(cur).mean())

np.fill_diagonal(D_km, 0) 

deg   = A_hybrid.sum(axis=1)
deg   = np.where(deg == 0, 1, deg)        
D_inv = np.diag(1.0 / np.sqrt(deg))
A_norm = D_inv @ A_hybrid @ D_inv

A_hat    = A_norm + np.eye(N_NODES)
A_tensor = torch.FloatTensor(A_hat).to(DEVICE)

deg_raw = A_hybrid.astype(bool).sum(axis=1)
print(f'Metode Adjacency    : Hybrid KNN (k={K}) + Gaussian (σ={SIGMA} km)')
print(f'Shape               : {A_hat.shape}')
print(f'Isolated nodes      : {(deg_raw == 0).sum()}  (target: 0)')
print(f'Avg degree          : {deg_raw.mean():.1f}')
print(f'Min / Max degree    : {deg_raw.min()} / {deg_raw.max()}')
print(f'Edge density        : {A_hybrid.astype(bool).sum() / (N_NODES*(N_NODES-1)):.3f}')
print(f'Weight range        : [{A_hybrid[A_hybrid>0].min():.4f}, {A_hybrid.max():.4f}]')


In [ ]:
degrees = A_hybrid.astype(bool).sum(axis=1)  
weights = A_hybrid[A_hybrid > 0]            

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].hist(degrees, bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Degree (Neighbors)')
axes[0].set_ylabel('Jumlah node')
axes[0].set_title(f'Degree Distribution - Hybrid KNN (k={K}) + Gaussian (σ={SIGMA}km)')

axes[1].hist(weights, bins=30, color='teal', edgecolor='white')
axes[1].set_xlabel('Weighted Edge (Gaussian)')
axes[1].set_ylabel('Edge Total')
axes[1].set_title('Weighted Edge Distribution')

plt.tight_layout()
plt.savefig('outputs/stgnn_degree_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Isolated nodes  : {(degrees == 0).sum()}')
print(f'Avg degree      : {degrees.mean():.1f}')
print(f'Min/Max degree  : {degrees.min()} / {degrees.max()}')
print(f'Weight range    : [{weights.min():.4f}, {weights.max():.4f}]')


In [ ]:
# Adjacency matrix
adj_df = pd.DataFrame(
    A_hybrid,
    index=NODE_LIST,
    columns=NODE_LIST
)

# Center of node
center = "KOTA BANDUNG"
neighbors = adj_df.loc[center]
neighbors = neighbors[neighbors > 0].sort_values(ascending=False)
selected_nodes = [center] + neighbors.index.tolist()
adj_subset = adj_df.loc[selected_nodes, selected_nodes].round(3)
adj_subset

In [ ]:
def get_original_knn(node_name, K=5):
    i = NODE_LIST.index(node_name)

    distances = D_km[i].copy()
    distances[i] = np.inf

    knn_idx = np.argsort(distances)[:K]

    return pd.DataFrame({
        "Neighbors": [NODE_LIST[j] for j in knn_idx],
        "Distance_km": [D_km[i, j] for j in knn_idx]
    })

display(get_original_knn("KOTA BANDUNG"))

##### Feature Engineering

In [ ]:
df = df.sort_values(['Kab/Kota', 'Year_Month']).reset_index(drop=True)

df['Month']          = df['Year_Month'].dt.month
df['Month_Sin']      = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_Cos']      = np.cos(2 * np.pi * df['Month'] / 12)

NODE_FEATURES = [
    'DIR_log', 
    'Month_Sin',       
    'Month_Cos',
    'temp_mean',      
    'humidity_mean',
    'precip_sum',
]
N_FEATURES = len(NODE_FEATURES)

print(f'Node features ({N_FEATURES}):')
for i, f in enumerate(NODE_FEATURES):
    print(f'  [{i}] {f}')

In [ ]:
df.dropna(inplace=True)

In [ ]:
# Pivot: (T, N, F) - timesteps x nodes x features
timestamps = sorted(df['Year_Month'].unique())
T = len(timestamps)

# Tensor (T, N, F)
X_raw = np.zeros((T, N_NODES, N_FEATURES), dtype=np.float32)

for t_idx, ts in enumerate(timestamps):
    df_t = df[df['Year_Month'] == ts].copy()
    for _, row in df_t.iterrows():
        n_idx = node2idx.get(row['Kab/Kota'])
        if n_idx is not None:
            X_raw[t_idx, n_idx, :] = row[NODE_FEATURES].values

print(f'X_raw shape: {X_raw.shape}  → (T={T}, N={N_NODES}, F={N_FEATURES})')

In [ ]:
# Train: Jan 2005 – Des 2017 | Valid: Jan 2018 – Des 2021 | Test: Jan 2022 – Des 2023
timestamps_pd = pd.to_datetime(timestamps)

train_mask = timestamps_pd < '2018-01-01'
valid_mask  = (timestamps_pd >= '2018-01-01') & (timestamps_pd < '2022-01-01')
test_mask   = (timestamps_pd >= '2022-01-01') & (timestamps_pd < '2024-01-01')

T_train = train_mask.sum()
T_valid  = valid_mask.sum()
T_test   = test_mask.sum()

print(f'Train: {T_train} timesteps ({timestamps_pd[train_mask].min()} -> {timestamps_pd[train_mask].max()})')
print(f'Valid : {T_valid} timesteps ({timestamps_pd[valid_mask].min()} -> {timestamps_pd[valid_mask].max()})')
print(f'Test  : {T_test} timesteps ({timestamps_pd[test_mask].min()} -> {timestamps_pd[test_mask].max()})')

# Scaler fit for train — reshape to 2D
X_train_flat = X_raw[train_mask].reshape(-1, N_FEATURES)
scaler = StandardScaler()
scaler.fit(X_train_flat)

# Transform
def scale_X(X_split):
    t, n, f = X_split.shape
    return scaler.transform(X_split.reshape(-1, f)).reshape(t, n, f)

X_train_sc = scale_X(X_raw[train_mask])
X_valid_sc  = scale_X(X_raw[valid_mask])
X_test_sc   = scale_X(X_raw[test_mask])

# Target: DIR_log
TARGET_IDX = 0
y_train_sc = X_train_sc[:, :, TARGET_IDX]
y_valid_sc  = X_valid_sc[:, :, TARGET_IDX]
y_test_sc   = X_test_sc[:, :, TARGET_IDX]

print(f'\nX_train_sc shape: {X_train_sc.shape}')
print(f'y_train_sc shape: {y_train_sc.shape}')

##### Dataset dan DataLoader

In [ ]:
class STGNNDataset(Dataset):
    """
    Sliding window dataset untuk STGNN.
    Input : X[t-window:t]  shape (window, N, F)
    Target: y[t]           shape (N,) — DIR_log semua node di timestep t
    """
    def __init__(self, X, y, window=12):
        self.X      = torch.FloatTensor(X)
        self.y      = torch.FloatTensor(y)
        self.window = window

    def __len__(self):
        return len(self.X) - self.window

    def __getitem__(self, idx):
        x_seq  = self.X[idx : idx + self.window]          # (window, N, F)
        y_next = self.y[idx + self.window]                 # (N,)
        return x_seq, y_next

WINDOW     = 12   
BATCH_SIZE = 16

train_ds = STGNNDataset(X_train_sc, y_train_sc, WINDOW)
valid_ds  = STGNNDataset(X_valid_sc,  y_valid_sc,  WINDOW)
test_ds   = STGNNDataset(X_test_sc,   y_test_sc,   WINDOW)

print(f'Train samples: {len(train_ds)}')
print(f'Valid samples: {len(valid_ds)}')
print(f'Test samples : {len(test_ds)}')

_tmp_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
x_batch, y_batch = next(iter(_tmp_dl))
print(f'\nx_batch shape: {x_batch.shape}  → (batch, window, N, F)')
print(f'y_batch shape: {y_batch.shape}  → (batch, N)')
del _tmp_dl  

##### Arsitektur STGNN

In [ ]:
class SpatialGCN(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.1):
        super().__init__()
        self.W_agg  = nn.Linear(in_features, out_features, bias=True)
        self.W_self = nn.Linear(in_features, out_features, bias=True)
        self.dropout = nn.Dropout(dropout)
        self.norm   = nn.LayerNorm(out_features)
    
    def forward(self, x, A):
        agg  = torch.matmul(A, x)             
        h_agg  = self.W_agg(agg)              
        h_self = self.W_self(x)                 
        
        out = F.gelu(h_agg + h_self)            
        out = self.dropout(out)
        out = self.norm(out)
        return out

In [ ]:
class GatedTCN(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, 
                 dilation=1, dropout=0.1):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        
        self.conv_tanh = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=self.padding, dilation=dilation
        )
        self.conv_sigma = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=self.padding, dilation=dilation
        )
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(out_channels)
        
        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )
    
    def forward(self, x):
        residual = self.residual(x)
        
        # Causal: trim right side
        h_tanh  = torch.tanh(self.conv_tanh(x)[:, :, :x.size(2)])
        h_sigma = torch.sigmoid(self.conv_sigma(x)[:, :, :x.size(2)])
        
        out = h_tanh * h_sigma
        out = out + residual
        out = self.dropout(out)
        
        # LayerNorm per timestep
        out = out.permute(0, 2, 1)  # (B*N, window, C)
        out = self.norm(out)
        out = out.permute(0, 2, 1)  # back to (B*N, C, window)
        
        return out

In [ ]:
class TemporalAttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
    
    def forward(self, x):
        x_t = x.permute(0, 2, 1)       # (B*N, window, C)
        scores = self.attn(x_t)         # (B*N, window, 1)
        weights = F.softmax(scores, dim=1)
        pooled = (x_t * weights).sum(dim=1)  # (B*N, C)
        return pooled

In [ ]:
class STGNNBlock(nn.Module):

    def __init__(self, in_features, gcn_hidden, tcn_hidden,
                 kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        self.gcn = SpatialGCN(in_features, gcn_hidden, dropout)
        self.tcn = GatedTCN(gcn_hidden, tcn_hidden, kernel_size, 
                                dilation, dropout)
        self.attn_pool = TemporalAttentionPooling(tcn_hidden)
        
        self.residual_proj = nn.Linear(in_features, tcn_hidden)
    
    def forward(self, x, A):
        batch, window, N, F = x.shape
        
        x_flat = x.reshape(batch * window, N, F)
        gcn_out = self.gcn(x_flat, A)
        gcn_out = gcn_out.reshape(batch, window, N, -1)
        
        gcn_h = gcn_out.shape[-1]
        x_tcn = gcn_out.permute(0, 2, 3, 1).reshape(batch * N, gcn_h, window)
        tcn_out = self.tcn(x_tcn)  
        
        pooled = self.attn_pool(tcn_out)  # (B*N, tcn_hidden)
        pooled = pooled.reshape(batch, N, -1)
        
        x_last = x[:, -1, :, :]  # (batch, N, F)
        residual = self.residual_proj(x_last)  # (batch, N, tcn_hidden)
        
        return pooled + residual

In [ ]:
class STGNN(nn.Module):

    def __init__(self, n_features, n_nodes, gcn_hidden=64, tcn_hidden=64,
                 mlp_hidden=64, kernel_size=3, dropout=0.15,
                 node_embed_dim=6):
        super().__init__()
        
        self.block = STGNNBlock(
            n_features, gcn_hidden, tcn_hidden,
            kernel_size=kernel_size, dilation=1, dropout=dropout
        )
        
        self.node_embed = nn.Embedding(n_nodes, node_embed_dim)
        
        mlp_input = tcn_hidden + node_embed_dim
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(mlp_hidden // 2, 1)
        )
        
        self.register_buffer('node_idx', torch.arange(n_nodes))
    
    def forward(self, x, A):
        """
        x: (batch, window, N, F) → (batch, N)
        """
        batch = x.shape[0]
        N = x.shape[2]
        
        h = self.block(x, A)  # (batch, N, tcn_hidden)
        
        n_emb = self.node_embed(self.node_idx)  # (N, node_embed_dim)
        n_emb = n_emb.unsqueeze(0).expand(batch, -1, -1)  # (batch, N, node_embed_dim)
        h = torch.cat([h, n_emb], dim=-1)  # (batch, N, tcn_hidden + node_embed_dim)
        
        # MLP per node
        pred = self.mlp(h).squeeze(-1)  # (batch, N)
        return pred

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_dataloaders(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, 
                          shuffle=True, generator=g)
    valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
    return train_dl, valid_dl, test_dl


def train_ablation_model(model, train_dl, valid_dl, A_tensor, DEVICE,
                          epochs=500, lr=5e-4, patience=60, 
                          save_path=None, label='Model',
                          verbose=False, log_every=10, return_history=False):
    _t0 = time.perf_counter()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=20
    )
    criterion = nn.MSELoss()
    
    best_valid = np.inf
    patience_c = 0
    history = {'train_loss': [], 'valid_loss': []}
    
    print(f'\n  {label}:')
    print(f'    Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
    
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_dl, optimizer, criterion, A_tensor, DEVICE)
        valid_loss, _, _ = eval_epoch(model, valid_dl, criterion, A_tensor, DEVICE)
        scheduler.step(valid_loss)
        
        history['train_loss'].append(train_loss)
        history['valid_loss'].append(valid_loss)
        
        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f'    Epoch {epoch:3d}/{epochs} | '
                  f'Train Loss: {train_loss:.5f} | '
                  f'Valid Loss: {valid_loss:.5f}')
        
        if valid_loss < best_valid:
            best_valid = valid_loss
            patience_c = 0
            if save_path:
                torch.save(model.state_dict(), save_path)
        else:
            patience_c += 1
            if patience_c >= patience:
                if verbose:
                    print(f'    Early stop at epoch {epoch}')
                break
    
    print(f'    Best valid loss: {best_valid:.5f} (stopped at epoch {epoch})')
    
    if save_path:
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))

    train_time_sec = time.perf_counter() - _t0
    print(f'    Train time: {train_time_sec:.1f}s')
    if return_history:
        history['train_time_sec'] = train_time_sec
        return model, history
    return model, train_time_sec

def evaluate_variant(model, train_dl, valid_dl, test_dl, A_tensor, DEVICE):
    criterion_eval = nn.MSELoss()
    
    def get_metrics(dl):
        _, pred_sc, tgt_sc = eval_epoch(model, dl, criterion_eval, A_tensor, DEVICE)
        pred = inv_target(pred_sc)
        tgt  = inv_target(tgt_sc)
        metrics = eval_metrics(tgt, pred, label='')
        return {**metrics, 'pred': pred, 'target': tgt}
    
    return {
        'train': get_metrics(train_dl),
        'valid': get_metrics(valid_dl),
        'test':  get_metrics(test_dl),
    }

##### Training Setup

In [ ]:
# Hyperparameter
GCN_HIDDEN  = 32
TCN_HIDDEN  = 64
MLP_HIDDEN  = 64
KERNEL_SIZE = 3
DROPOUT     = 0.1
LR          = 5e-4
EPOCHS      = 500
PATIENCE    = 60
NODE_EMBED  = 6

set_seed(42)
train_dl, valid_dl, test_dl = make_dataloaders(42)

model = STGNN(
    n_features     = N_FEATURES,
    n_nodes        = N_NODES,
    gcn_hidden     = GCN_HIDDEN,
    tcn_hidden     = TCN_HIDDEN,
    mlp_hidden     = MLP_HIDDEN,
    kernel_size    = KERNEL_SIZE,
    dropout        = DROPOUT,
    node_embed_dim = NODE_EMBED,
).to(DEVICE)

# Jumlah parameter
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {model.__class__.__name__}')
print(f'Jumlah parameter: {n_params:,}')
print(model)

##### Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, A, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)   # (batch, window, N, F)
        y_batch = y_batch.to(device)   # (batch, N)

        optimizer.zero_grad()
        pred = model(x_batch, A)       # (batch, N)
        loss = criterion(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * x_batch.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, criterion, A, device):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        pred = model(x_batch, A)

        if pred.ndim == 3:
            pred = pred[:, -1, :]

        loss = criterion(pred, y_batch)
        total_loss += loss.item() * x_batch.size(0)

        all_preds.append(pred.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    preds = np.concatenate(all_preds, axis=0)
    targets = np.concatenate(all_targets, axis=0)

    return avg_loss, preds, targets

In [ ]:
# Training
model, history = train_ablation_model(
    model, train_dl, valid_dl, A_tensor, DEVICE,
    epochs         = EPOCHS,
    lr             = LR,
    patience       = PATIENCE,
    save_path      = 'outputs/stgnn_standalone_seed42.pt',
    label          = 'STGNN',
    verbose        = True,     
    log_every      = 10,         
    return_history = True,      
)

In [ ]:
# Plot training curve
plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['valid_loss'], label='Valid Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('STGNN Training Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/stgnn_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

##### Evaluasi

In [ ]:
from sklearn.metrics import mean_absolute_error

# Metric functions
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))

def eval_metrics(tgt, pred, label=''):
    y, p = tgt.flatten(), pred.flatten()
    r = rmse(y, p)
    m = mean_absolute_error(y, p)
    if label:
        print(f'  {label:<8} RMSE={r:.6f}  MAE={m:.6f}')
    return {'RMSE': r, 'MAE': m}

# Inverse transform
mean0 = scaler.mean_[TARGET_IDX]
std0  = scaler.scale_[TARGET_IDX]

def inv_target(arr_sc):
    return arr_sc * std0 + mean0

In [ ]:
def to_dir(log_arr):
    return np.expm1(log_arr)         
    
def _rmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.sqrt(np.mean((a - b) ** 2))
 
def _mae(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.mean(np.abs(a - b))
 
def dual_metrics(target_log, pred_log):

    t_log = np.asarray(target_log, float).ravel()
    p_log = np.asarray(pred_log,   float).ravel()
    t_dir, p_dir = to_dir(t_log), to_dir(p_log)
    return {
        'RMSE_log': _rmse(t_log, p_log),
        'MAE_log':  _mae(t_log,  p_log),
        'RMSE_DIR': _rmse(t_dir, p_dir),
        'MAE_DIR':  _mae(t_dir,  p_dir),
    }
 
def dual_metrics_per_node(target_log, pred_log, node_list):
    t_log = np.asarray(target_log, float)      # (T, N)
    p_log = np.asarray(pred_log,   float)
    t_dir, p_dir = to_dir(t_log), to_dir(p_log)
    e_log, e_dir = p_log - t_log, p_dir - t_dir
    return pd.DataFrame({
        'Kab/Kota'       : node_list,
        'Mean_DIR_aktual': t_dir.mean(axis=0),          # konteks incidence
        'RMSE_log'       : np.sqrt(np.mean(e_log**2, axis=0)),
        'MAE_log'        : np.mean(np.abs(e_log), axis=0),
        'RMSE_DIR'       : np.sqrt(np.mean(e_dir**2, axis=0)),
        'MAE_DIR'        : np.mean(np.abs(e_dir), axis=0),
    })
 
print("Helper dual-scale siap: dual_metrics(), dual_metrics_per_node()")

In [ ]:
result = evaluate_variant(model, train_dl, valid_dl, test_dl, A_tensor, DEVICE)

print('\nSTGNN EVALUATION')
for split in ['train', 'valid', 'test']:
    r = result[split]
    print(f"  {split:<6} RMSE={r['RMSE']:.6f}  MAE={r['MAE']:.6f}")

In [ ]:
np.savez(
    "outputs/predictions_full_seed42.npz",
    pred_test_orig=result["test"]["pred"],
    target_test_orig=result["test"]["target"],
)

print("Prediction saved")

#### Save Model

In [ ]:
torch.save(model.state_dict(), "best_model.pt")

#### Mapping

In [ ]:
import json
import urllib.request
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.patches import PathPatch, Patch
from matplotlib.path import Path

# Sumber: Humanitarian Data Exchange (HDX)
GEOJSON_LOCAL = 'data/indonesia_kabupaten.geojson'
 
if not os.path.exists(GEOJSON_LOCAL):
    print('Downloading geojson dari HDX (sekali saja)...')
    os.makedirs(os.path.dirname(GEOJSON_LOCAL), exist_ok=True)
    urllib.request.urlretrieve(GEOJSON_URL, GEOJSON_LOCAL)
    print(f'Tersimpan di: {GEOJSON_LOCAL}')
 
with open(GEOJSON_LOCAL) as f:
    gj_all = json.load(f)

In [ ]:
java_prov_ids = {'31', '32', '33', '34', '35', '36'}
java_features = [f for f in gj_all['features']
                 if f['properties']['prov_id'] in java_prov_ids]
geom_lookup   = {f['properties']['name']: f['geometry'] for f in java_features}
print(f'Java features from geojson: {len(java_features)}')

In [ ]:
missing = [n for n in NODE_LIST if n not in geom_lookup]
if missing:
    raise ValueError(f'Kabupaten berikut tidak ada di geojson: {missing}')
print(f'Semua {len(NODE_LIST)} kabupaten match dengan geojson')
 

In [ ]:
df_centroid = pd.read_csv('data/df_centroid.csv')
df_geo = df_centroid.set_index('Kab/Kota').loc[NODE_LIST].reset_index()
lons = df_geo['longitude'].values
lats = df_geo['latitude'].values

In [ ]:
def _geom_to_path(geom):
    verts, codes = [], []
    polys = geom['coordinates'] if geom['type'] == 'MultiPolygon' else [geom['coordinates']]
    for poly in polys:
        for ring in poly:
            if len(ring) < 3:
                continue
            r = np.array(ring)
            verts.extend(r.tolist())
            codes.extend([Path.MOVETO]
                         + [Path.LINETO] * (len(r) - 2)
                         + [Path.CLOSEPOLY])
    return Path(verts, codes)
 
paths = [_geom_to_path(geom_lookup[n]) for n in NODE_LIST]

In [ ]:
BBOX      = [105.0, 115.0, -9.0, -5.5]   
SEA_COLOR = '#eaf4fb'                    
OUT_DIR   = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
 
def _draw_choropleth(ax, values, cmap, norm, ec='white', lw=0.4):
    """Helper: gambar 119 polygon kabupaten dengan warna sesuai values."""
    for i, path in enumerate(paths):
        ax.add_patch(PathPatch(path, facecolor=cmap(norm(values[i])),
                               edgecolor=ec, linewidth=lw))
 
def _setup_axes(ax, title):
    ax.set_xlim(BBOX[0], BBOX[1]); ax.set_ylim(BBOX[2], BBOX[3])
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(title, fontsize=11.5, fontweight='bold')
    ax.grid(True, alpha=0.25); ax.set_aspect('equal'); ax.set_facecolor(SEA_COLOR)

In [ ]:
# Prediction (seed 42)
pred_test_orig   = result['test']['pred']
target_test_orig = result['test']['target']

print('Loaded predictions from evaluation result')
print(f'  pred_test_orig shape  : {pred_test_orig.shape}')
print(f'  target_test_orig shape: {target_test_orig.shape}')


In [ ]:
# =============================================================================
# MAP 1: Spasial Mapping RMSE Test  
rmse_per_node = np.sqrt(np.mean((target_test_orig - pred_test_orig) ** 2, axis=0))
df_geo['Test_RMSE'] = rmse_per_node
 
fig, ax = plt.subplots(figsize=(15, 7))
norm = Normalize(vmin=np.percentile(rmse_per_node, 5),
                 vmax=np.percentile(rmse_per_node, 95))
_draw_choropleth(ax, rmse_per_node, plt.cm.YlOrRd, norm)
 
sm = plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd, norm=norm); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label('RMSE Test (DIR_log)', fontsize=11)
 
worst3 = df_geo.nlargest(3, 'Test_RMSE')
best3  = df_geo.nsmallest(3, 'Test_RMSE')
def _annot(row, color, dy):
    ax.annotate(f"{row['Kab/Kota']}\n{row['Test_RMSE']:.3f}",
                xy=(row['longitude'], row['latitude']),
                xytext=(0, dy), textcoords='offset points',
                ha='center', fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.25', fc='white', ec=color, lw=0.8, alpha=0.92),
                arrowprops=dict(arrowstyle='-', color=color, lw=0.6))
for _, r in worst3.iterrows(): _annot(r, 'darkred', 18)
for _, r in best3.iterrows():  _annot(r, 'darkgreen', -28)
 
ax.set_xlim(BBOX[0], BBOX[1]); ax.set_ylim(BBOX[2], BBOX[3])
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(
    f'Spatial Mapping of Test RMSE by Regency/City - STGNN\n'
    f'Mean={rmse_per_node.mean():.3f} | Median={np.median(rmse_per_node):.3f} | '
    f'Min={rmse_per_node.min():.3f} | Max={rmse_per_node.max():.3f}',
    fontsize=11.5)
ax.grid(True, alpha=0.25); ax.set_aspect('equal'); ax.set_facecolor(SEA_COLOR)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/map_rmse_choropleth.png', dpi=140, bbox_inches='tight')
plt.show()
 
df_geo[['Provinsi','Kab/Kota','latitude','longitude','Test_RMSE']] \
    .sort_values('Test_RMSE') \
    .to_csv(f'{OUT_DIR}/rmse_per_kabkota.csv', index=False)
 
print('\nTOP-5 RMSE terkecil (model terbaik):')
print(df_geo.nsmallest(5,'Test_RMSE')[['Kab/Kota','Test_RMSE']].to_string(index=False))
print('\nTOP-5 RMSE terbesar (model terlemah):')
print(df_geo.nlargest(5,'Test_RMSE')[['Kab/Kota','Test_RMSE']].to_string(index=False))

In [ ]:
# MAP 2: Actual vs Prediction Risk Class 
from sklearn.metrics import cohen_kappa_score, confusion_matrix

mean_pred   = pred_test_orig.mean(axis=0)
mean_actual = target_test_orig.mean(axis=0)

q33f, q66f = np.percentile(mean_actual, [33.33, 66.67])
risk_a = _classify(mean_actual, q33f, q66f)
risk_p = _classify(mean_pred,   q33f, q66f)

agree_full      = (risk_a == risk_p).mean() * 100
kappa_quadratic = cohen_kappa_score(risk_a, risk_p, weights='quadratic')
kappa_linear    = cohen_kappa_score(risk_a, risk_p, weights='linear')
kappa_unweight  = cohen_kappa_score(risk_a, risk_p)

# Confusion matrix for boundary cases analysis 
cm = confusion_matrix(risk_a, risk_p, labels=[0, 1, 2])
boundary_jump = cm[0, 2] + cm[2, 0]
adjacent_err  = cm[0, 1] + cm[1, 0] + cm[1, 2] + cm[2, 1]

def _kappa_interpret(k):
    if k < 0.00:  return "Poor"
    if k < 0.21:  return "Slight"
    if k < 0.41:  return "Fair"
    if k < 0.61:  return "Moderate"
    if k < 0.81:  return "Substantial"
    return "Almost Perfect"

print("=" * 60)
print(f"AGREEMENT METRICS — Seed 42 (canonical)")
print("=" * 60)
print(f"Percentage Agreement      : {agree_full:.2f}%")
print(f"Cohen's Kappa (quadratic) : {kappa_quadratic:.4f}  → {_kappa_interpret(kappa_quadratic)}")
print(f"Cohen's Kappa (linear)    : {kappa_linear:.4f}  → {_kappa_interpret(kappa_linear)}")
print(f"Cohen's Kappa (unweight)  : {kappa_unweight:.4f}  → {_kappa_interpret(kappa_unweight)}")
print(f"Misclassified             : {(~(risk_a == risk_p)).sum()} dari {len(risk_a)}")
print(f"  - Adjacent (1 kelas)    : {adjacent_err}")
print(f"  - Boundary (Low↔High)   : {boundary_jump}")
print("-" * 60)
print("Confusion Matrix:")
print("            Pred_Low  Pred_Mod  Pred_High")
print(f"Actual_Low    {cm[0,0]:4d}     {cm[0,1]:4d}     {cm[0,2]:4d}")
print(f"Actual_Mod    {cm[1,0]:4d}     {cm[1,1]:4d}     {cm[1,2]:4d}")
print(f"Actual_High   {cm[2,0]:4d}     {cm[2,1]:4d}     {cm[2,2]:4d}")
print("=" * 60)

t0 = pd.to_datetime(test_timestamps[0]).strftime('%b %Y')
t1 = pd.to_datetime(test_timestamps[-1]).strftime('%b %Y')

fig, axes = plt.subplots(1, 2, figsize=(19, 7))
for ax, risk_arr, title in [
    (axes[0], risk_a, f'Actual - Mean {t0} s.d. {t1}'),
    (axes[1], risk_p, f'STGNN Prediction - Mean {t0} s.d. {t1}'),
]:
    _draw_choropleth(ax, risk_arr, risk_cmap, risk_norm)
    counts = [(risk_arr == k).sum() for k in range(3)]
    handles = [Patch(facecolor=risk_cmap.colors[k], edgecolor='black',
                     label=f'{risk_labels[k]} (n={counts[k]})') for k in range(3)]
    ax.legend(handles=handles, title='Risk Category',
              loc='lower right', fontsize=9, title_fontsize=10)
    _setup_axes(ax, title)

plt.suptitle(
    f'Pemetaan Risiko DBD Rata-rata Periode Test\n'
    f'Risk-class Agreement: {agree_full:.1f}%  ·  '
    f"Weighted Kappa: {kappa_quadratic:.3f} ({_kappa_interpret(kappa_quadratic)})",
    fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/map_risk_mean_choropleth.png', dpi=140, bbox_inches='tight')
plt.show()

df_risk_full = pd.DataFrame({
    'Provinsi'        : df_geo['Provinsi'].values,
    'Kab/Kota'        : NODE_LIST,
    'latitude'        : lats,
    'longitude'       : lons,
    'DIR_actual_mean' : mean_actual,
    'DIR_pred_mean'   : mean_pred,
    'Risk_Actual'     : [risk_labels[k].split()[0] for k in risk_a],
    'Risk_Predicted'  : [risk_labels[k].split()[0] for k in risk_p],
    'Match'           : (risk_a == risk_p),
})
df_risk_full.to_csv(f'{OUT_DIR}/risk_classification_full.csv', index=False)
print(f'\nMean-period risk agreement: {agree_full:.1f}%')
print(f'Weighted Kappa: {kappa_quadratic:.3f}')
print(f'Mismatch: {(~df_risk_full["Match"]).sum()} of {len(NODE_LIST)} kabupaten')

#### ABLATION STUDY - Multi-Seed for Robustness Test

In [ ]:
class TCNOnlyBlock(nn.Module):

    def __init__(self, in_features, tcn_hidden, kernel_size=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(in_features, tcn_hidden)
        self.tcn        = GatedTCN(tcn_hidden, tcn_hidden, kernel_size, 
                                       dilation=1, dropout=dropout)
        self.attn_pool  = TemporalAttentionPooling(tcn_hidden)
        self.residual_proj = nn.Linear(in_features, tcn_hidden)
    
    def forward(self, x, A):  
        batch, window, N, F = x.shape
        
        x_proj = self.input_proj(x)  # (batch, window, N, tcn_hidden)
        
        # Reshape for TCN: (batch*N, tcn_hidden, window)
        x_tcn = x_proj.permute(0, 2, 3, 1).reshape(batch * N, -1, window)
        tcn_out = self.tcn(x_tcn)
        
        # Attention pooling
        pooled = self.attn_pool(tcn_out)
        pooled = pooled.reshape(batch, N, -1)
        
        x_last = x[:, -1, :, :]
        residual = self.residual_proj(x_last)
        
        return pooled + residual
 
 
class GCNOnlyBlock(nn.Module):
    def __init__(self, in_features, gcn_hidden, out_hidden, dropout=0.1):
        super().__init__()
        self.gcn1 = SpatialGCN(in_features, gcn_hidden, dropout)
        self.gcn2 = SpatialGCN(gcn_hidden, out_hidden, dropout)
        self.residual_proj = nn.Linear(in_features, out_hidden)
    
    def forward(self, x, A):
        x_last = x[:, -1, :, :]  # (batch, N, F)
        
        h = self.gcn1(x_last, A)
        h = self.gcn2(h, A)
        
        residual = self.residual_proj(x_last)
        return h + residual
 
 
class STGNN_TCN_only(nn.Module):
    """STGNN tanpa GCN — temporal model murni."""
    def __init__(self, n_features, n_nodes, tcn_hidden=64, mlp_hidden=64,
                 kernel_size=3, dropout=0.1, node_embed_dim=7):
        super().__init__()
        self.block = TCNOnlyBlock(n_features, tcn_hidden, kernel_size, dropout)
        self.node_embed = nn.Embedding(n_nodes, node_embed_dim)
        
        mlp_input = tcn_hidden + node_embed_dim
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(mlp_hidden // 2, 1)
        )
        self.register_buffer('node_idx', torch.arange(n_nodes))
    
    def forward(self, x, A):
        batch = x.shape[0]
        h = self.block(x, A)
        n_emb = self.node_embed(self.node_idx).unsqueeze(0).expand(batch, -1, -1)
        h = torch.cat([h, n_emb], dim=-1)
        return self.mlp(h).squeeze(-1)
 
 
class STGNN_GCN_only(nn.Module):
    """STGNN tanpa TCN — spatial model murni."""
    def __init__(self, n_features, n_nodes, gcn_hidden=64, out_hidden=64,
                 mlp_hidden=64, dropout=0.1, node_embed_dim=7):
        super().__init__()
        self.block = GCNOnlyBlock(n_features, gcn_hidden, out_hidden, dropout)
        self.node_embed = nn.Embedding(n_nodes, node_embed_dim)
        
        mlp_input = out_hidden + node_embed_dim
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(mlp_hidden // 2, 1)
        )
        self.register_buffer('node_idx', torch.arange(n_nodes))
    
    def forward(self, x, A):
        batch = x.shape[0]
        h = self.block(x, A)
        n_emb = self.node_embed(self.node_idx).unsqueeze(0).expand(batch, -1, -1)
        h = torch.cat([h, n_emb], dim=-1)
        return self.mlp(h).squeeze(-1)
 

In [ ]:
class STGNNBlock_NoAttn(nn.Module):
    def __init__(self, in_features, gcn_hidden, tcn_hidden,
                 kernel_size=3, dropout=0.1):
        super().__init__()
        self.gcn = SpatialGCN(in_features, gcn_hidden, dropout)
        self.tcn = GatedTCN(gcn_hidden, tcn_hidden, kernel_size,
                            dilation=1, dropout=dropout)
        self.residual_proj = nn.Linear(in_features, tcn_hidden)
    
    def forward(self, x, A):
        batch, window, N, F = x.shape
        
        x_flat = x.reshape(batch * window, N, F)
        gcn_out = self.gcn(x_flat, A)
        gcn_out = gcn_out.reshape(batch, window, N, -1)
        
        gcn_h = gcn_out.shape[-1]
        x_tcn = gcn_out.permute(0, 2, 3, 1).reshape(batch * N, gcn_h, window)
        tcn_out = self.tcn(x_tcn)  # (batch*N, tcn_hidden, window)
        
        pooled = tcn_out[:, :, -1]              # (batch*N, tcn_hidden)
        pooled = pooled.reshape(batch, N, -1)   # (batch, N, tcn_hidden)
        
        x_last = x[:, -1, :, :]
        residual = self.residual_proj(x_last)
        
        return pooled + residual


class STGNN_NoAttn(nn.Module):
    def __init__(self, n_features, n_nodes, gcn_hidden=64, tcn_hidden=64,
                 mlp_hidden=64, kernel_size=3, dropout=0.1, node_embed_dim=6):
        super().__init__()
        self.block = STGNNBlock_NoAttn(n_features, gcn_hidden, tcn_hidden,
                                        kernel_size, dropout)
        self.node_embed = nn.Embedding(n_nodes, node_embed_dim)
        
        mlp_input = tcn_hidden + node_embed_dim
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(mlp_hidden // 2, 1)
        )
        self.register_buffer('node_idx', torch.arange(n_nodes))
    
    def forward(self, x, A):
        batch = x.shape[0]
        h = self.block(x, A)
        n_emb = self.node_embed(self.node_idx).unsqueeze(0).expand(batch, -1, -1)
        h = torch.cat([h, n_emb], dim=-1)
        return self.mlp(h).squeeze(-1)

In [ ]:
import torch.nn as nn
import random

SEEDS = [42, 123, 456, 789, 2024]

all_seed_results = {}  # {seed: {variant: {metrics}}}

for seed in SEEDS:
    print(f'\n{"="*70}')
    print(f'SEED = {seed}')
    print(f'{"="*70}')
    
    seed_results = {}
    
    # 1. STGNN-Full
    set_seed(seed)
    train_dl, valid_dl, test_dl = make_dataloaders(seed)
    model_full_new = STGNN(
        n_features     = N_FEATURES,
        n_nodes        = N_NODES,
        gcn_hidden     = GCN_HIDDEN,
        tcn_hidden     = TCN_HIDDEN,
        mlp_hidden     = MLP_HIDDEN,
        kernel_size    = KERNEL_SIZE,
        dropout        = DROPOUT,
        node_embed_dim = NODE_EMBED,
    ).to(DEVICE)
    
    model_full_new, _ = train_ablation_model(
        model_full_new, train_dl, valid_dl, A_tensor, DEVICE,
        epochs=EPOCHS, lr=LR, patience=PATIENCE,
        save_path=f'outputs/ablation_full_seed{seed}.pt',
        label='STGNN-Full')
    res_full = evaluate_variant(
        model_full_new, train_dl, valid_dl, test_dl, A_tensor, DEVICE
    )
    seed_results['STGNN-Full'] = res_full
    np.savez(
        f'outputs/ablation_full_predictions_seed{seed}.npz',
        pred_test_orig=res_full['test']['pred'],
        target_test_orig=res_full['test']['target'],
    )
    
    # 2. STGNN-TCN only
    set_seed(seed)
    train_dl, valid_dl, test_dl = make_dataloaders(seed)
    model_tcn_new = STGNN_TCN_only(
        n_features     = N_FEATURES,
        n_nodes        = N_NODES,
        tcn_hidden     = TCN_HIDDEN,
        mlp_hidden     = MLP_HIDDEN,
        kernel_size    = KERNEL_SIZE,
        dropout        = DROPOUT,
        node_embed_dim = NODE_EMBED,
    ).to(DEVICE)
    
    model_tcn_new, _ = train_ablation_model(
        model_tcn_new, train_dl, valid_dl, A_tensor, DEVICE,
        epochs=EPOCHS, lr=LR, patience=PATIENCE,
        save_path=f'outputs/ablation_tcn_seed{seed}.pt',
        label='STGNN-TCN only'
    )
    seed_results['STGNN-TCN_only'] = evaluate_variant(
        model_tcn_new, train_dl, valid_dl, test_dl, A_tensor, DEVICE
    )
    np.savez(
        f'outputs/ablation_tcn_predictions_seed{seed}.npz',
        pred_test_orig=seed_results['STGNN-TCN_only']['test']['pred'],
        target_test_orig=seed_results['STGNN-TCN_only']['test']['target'],
    )
    
    # 3. STGNN-GCN only
    set_seed(seed)
    train_dl, valid_dl, test_dl = make_dataloaders(seed)
    model_gcn_new = STGNN_GCN_only(
        n_features     = N_FEATURES,
        n_nodes        = N_NODES,
        gcn_hidden     = GCN_HIDDEN,
        out_hidden     = TCN_HIDDEN,
        mlp_hidden     = MLP_HIDDEN,
        dropout        = DROPOUT,
        node_embed_dim = NODE_EMBED,
    ).to(DEVICE)
    
    model_gcn_new, _ = train_ablation_model(
        model_gcn_new, train_dl, valid_dl, A_tensor, DEVICE,
        epochs=EPOCHS, lr=LR, patience=PATIENCE,
        save_path=f'outputs/ablation_gcn_seed{seed}.pt',
        label='STGNN-GCN only'
    )
    seed_results['STGNN-GCN_only'] = evaluate_variant(
        model_gcn_new, train_dl, valid_dl, test_dl, A_tensor, DEVICE
    )
    np.savez(
        f'outputs/ablation_gcn_predictions_seed{seed}.npz',
        pred_test_orig=seed_results['STGNN-GCN_only']['test']['pred'],
        target_test_orig=seed_results['STGNN-GCN_only']['test']['target'],
    )

    #  4. STGNN-Full-MeanPool
    set_seed(seed)
    train_dl, valid_dl, test_dl = make_dataloaders(seed)
    model_mp_new = STGNN_NoAttn(
        n_features     = N_FEATURES,
        n_nodes        = N_NODES,
        gcn_hidden     = GCN_HIDDEN,
        tcn_hidden     = TCN_HIDDEN,
        mlp_hidden     = MLP_HIDDEN,
        kernel_size    = KERNEL_SIZE,
        dropout        = DROPOUT,
        node_embed_dim = NODE_EMBED,
    ).to(DEVICE)
    
    model_mp_new, _ = train_ablation_model(
        model_mp_new, train_dl, valid_dl, A_tensor, DEVICE,
        epochs=EPOCHS, lr=LR, patience=PATIENCE,
        save_path=f'outputs/ablation_meanpool_seed{seed}.pt',
        label='STGNN_NoAttn'
    )
    seed_results['STGNN_NoAttn'] = evaluate_variant(
        model_mp_new, train_dl, valid_dl, test_dl, A_tensor, DEVICE
    )
    np.savez(
        f'outputs/ablation_meanpool_predictions_seed{seed}.npz',
        pred_test_orig=seed_results['STGNN_NoAttn']['test']['pred'],
        target_test_orig=seed_results['STGNN_NoAttn']['test']['target'],
    )
    
    all_seed_results[seed] = seed_results


variants = ['STGNN-TCN_only', 'STGNN-GCN_only', 'STGNN_NoAttn', 'STGNN-Full']
metrics_to_track = ['RMSE', 'MAE']

summary_rows = []
for variant in variants:
    row = {'Model': variant}
    
    for metric in metrics_to_track:
        # Collect values across seeds
        values = [all_seed_results[seed][variant]['test'][metric] 
                  for seed in SEEDS]
        
        mean_val = np.mean(values)
        std_val  = np.std(values)
        
        row[f'{metric}_mean'] = mean_val
        row[f'{metric}_std']  = std_val
        row[f'{metric}_str']  = f'{mean_val:.4f} ± {std_val:.4f}'
    
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

print('\n' + '=' * 90)
print(f'ABLATION ROBUSTNESS — {len(SEEDS)} seeds: {SEEDS}')
print('=' * 90)

display_cols = ['Model'] + [f'{m}_str' for m in metrics_to_track]
display_df = summary_df[display_cols].copy()
display_df.columns = ['Model'] + metrics_to_track
print(display_df.to_string(index=False))

raw_rows = []
for seed in SEEDS:
    for variant in variants:
        row = {'Seed': seed, 'Model': variant}
        for metric in metrics_to_track:
            row[metric] = all_seed_results[seed][variant]['test'][metric]
        raw_rows.append(row)

raw_df = pd.DataFrame(raw_rows)
raw_df.to_csv('outputs/ablation_robustness_raw.csv', index=False)
summary_df.to_csv('outputs/ablation_robustness_summary.csv', index=False)
print('\nSaved: ablation_robustness_raw.csv & ablation_robustness_summary.csv')

#### Per-District Errors STGNN-Full (n=119)

In [ ]:
import numpy as np, pandas as pd

OUT_DIR = 'outputs'
N = N_NODES
assert N == 119, f'Expected 119 districts, got {N}'

frames = []
for seed in SEEDS:
    npz = np.load(f'{OUT_DIR}/ablation_full_predictions_seed{seed}.npz')
    p   = npz['pred_test_orig']            # (12, 119)
    y   = npz['target_test_orig']          # (12, 119)
    T   = p.shape[0]                        # 12
    err = (p - y).ravel()                   # C-order: idx = t*N + node
    frames.append(pd.DataFrame({
        'Model'    : 'STGNN-Full',
        'Seed'     : seed,
        'Kab/Kota' : np.tile(NODE_LIST, T),
        'month_idx': np.repeat(np.arange(T), N),
        'abs_err'  : np.abs(err),
        'sq_err'   : err ** 2,
    }))

stgnn_pd_df = pd.concat(frames, ignore_index=True)
stgnn_pd_df.to_csv(f'{OUT_DIR}/stgnn_full_per_district_raw.csv', index=False)
print(f'Saved stgnn_full_per_district_raw.csv  rows={len(stgnn_pd_df):,}  '
      f'(= {len(SEEDS)} seed × 12 bulan × {N} distrik)')

In [ ]:
variants = ['STGNN-Full', 'STGNN-TCN_only', 'STGNN-GCN_only', 'STGNN_NoAttn']

# Per-seed breakdown
print('=' * 75)
print(f'{"Seed":<6} {"Variant":<22} {"RMSE":>8} {"MAE":>8} {"ΔRMSE vs Full":>14}')
print('=' * 75)

for seed in SEEDS:
    full_rmse = all_seed_results[seed]['STGNN-Full']['test']['RMSE']
    for variant in variants:
        r = all_seed_results[seed][variant]['test']
        diff = r['RMSE'] - full_rmse  # positif = Full lebih baik
        marker = ' ← best' if variant == 'STGNN-Full' else ''
        print(f"{seed:<6} {variant:<22} {r['RMSE']:>8.4f} {r['MAE']:>8.4f} {diff:>+14.4f}{marker}")
    print('-' * 75)

# Summary mean ± std
print('\n' + '=' * 55)
print('SUMMARY MEAN ± STD (5 seeds)')
print('=' * 55)
print(f'{"Variant":<22} {"RMSE":>16} {"MAE":>16}')
print('-' * 55)

for variant in variants:
    rmse_vals = [all_seed_results[s][variant]['test']['RMSE'] for s in SEEDS]
    mae_vals  = [all_seed_results[s][variant]['test']['MAE']  for s in SEEDS]
    print(f"{variant:<22} "
          f"{np.mean(rmse_vals):.4f} ± {np.std(rmse_vals):.4f}  "
          f"{np.mean(mae_vals):.4f} ± {np.std(mae_vals):.4f}")

print('\n' + '=' * 55)
print('Contribution of Components (Mean for 5 seeds)')
print('=' * 55)

full_rmse_mean = np.mean([all_seed_results[s]['STGNN-Full']['test']['RMSE'] for s in SEEDS])

for variant in ['STGNN-TCN_only', 'STGNN-GCN_only', 'STGNN_NoAttn']:
    v_rmse_mean = np.mean([all_seed_results[s][variant]['test']['RMSE'] for s in SEEDS])
    delta = v_rmse_mean - full_rmse_mean
    pct   = delta / full_rmse_mean * 100
    komponen = {
        'STGNN-TCN_only':    'GCN Contributions (Full - TCN_only)',
        'STGNN-GCN_only':    'TCN Contributions (Full - GCN_only)',
        'STGNN_NoAttn':'TCN Contributions* (Full - STGNN_NoAttn)',
    }[variant]
    print(f"{komponen}: ΔRMSE={delta:+.4f} ({pct:+.1f}%)")

In [ ]:
full_mae_mean = np.mean([all_seed_results[s]['STGNN-Full']['test']['MAE'] for s in SEEDS])
for variant in ['STGNN-TCN_only', 'STGNN-GCN_only', 'STGNN_NoAttn']:
    v_mae_mean = np.mean([all_seed_results[s][variant]['test']['MAE'] for s in SEEDS])
    delta_mae = v_mae_mean - full_mae_mean
    pct_mae   = delta_mae / full_mae_mean * 100
    print(f"{komponen}: ΔMAE={delta_mae:+.4f} ({pct_mae:+.1f}%)")

In [ ]:
def extract_test_predictions(model, test_dl, A_tensor, device):
    model.eval()
    preds, targets = [], []
    
    with torch.no_grad():
        for X_batch, y_batch in test_dl:
            X_batch = X_batch.to(device)
            out = model(X_batch, A_tensor)         # shape (B, N) atau (B, N, 1)

            if out.dim() == 3 and out.shape[-1] == 1:
                out = out.squeeze(-1)
            if y_batch.dim() == 3 and y_batch.shape[-1] == 1:
                y_batch = y_batch.squeeze(-1)
            
            preds.append(out.cpu().numpy())
            targets.append(y_batch.numpy())
    
    pred_scaled   = np.concatenate(preds,   axis=0)   # (T_test, N)
    target_scaled = np.concatenate(targets, axis=0)   # (T_test, N)
    
    pred_orig   = inv_target(pred_scaled)
    target_orig = inv_target(target_scaled)
    
    return pred_orig, target_orig

In [ ]:
for seed in SEEDS:
    print(f'Extracting seed={seed}...')
    
    set_seed(seed)
    train_dl, valid_dl, test_dl = make_dataloaders(seed)
    
    model = STGNN(
        n_features     = N_FEATURES,
        n_nodes        = N_NODES,
        gcn_hidden     = GCN_HIDDEN,
        tcn_hidden     = TCN_HIDDEN,
        mlp_hidden     = MLP_HIDDEN,
        kernel_size    = KERNEL_SIZE,
        dropout        = DROPOUT,
        node_embed_dim = NODE_EMBED,
    ).to(DEVICE)
    
    # Load
    ckpt_path = f'outputs/ablation_full_seed{seed}.pt'
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    
    # Extract predictions
    pred_orig, target_orig = extract_test_predictions(
        model, test_dl, A_tensor, DEVICE
    )
    
    np.savez(
        f'outputs/predictions_full_seed{seed}.npz',
        pred_test_orig=pred_orig,
        target_test_orig=target_orig,
    )
    print(f'  seed={seed}: pred shape={pred_orig.shape}')

print('\nAll predictions extracted!')

In [ ]:
variants = ['STGNN-Full', 'STGNN-TCN_only', 'STGNN-GCN_only', 'STGNN_NoAttn']

summary_rows = []
for variant in variants:
    per_seed = []
    for seed in SEEDS:
        d = all_seed_results[seed][variant]['test']
        per_seed.append(dual_metrics(d['target'], d['pred']))
    row = {'Model': variant}
    for key in ['RMSE_log', 'MAE_log', 'RMSE_DIR', 'MAE_DIR']:
        vals = np.array([ps[key] for ps in per_seed])
        row[f'{key}_mean'] = vals.mean()
        row[f'{key}_std']  = vals.std()
        nd = 4 if key.endswith('log') else 2
        row[f'{key}'] = f"{vals.mean():.{nd}f} ± {vals.std():.{nd}f}"
    summary_rows.append(row)
 
df_ablation_dual = pd.DataFrame(summary_rows)
df_ablation_dual.to_csv('outputs/ablation_dual_scale_summary.csv', index=False)
 
print("=" * 84)
print(f"ABLATION × MULTI-SEED ({len(SEEDS)} seeds) - mean ± std, two scales")
print("=" * 84)
show = df_ablation_dual[['Model', 'RMSE_log', 'MAE_log', 'RMSE_DIR', 'MAE_DIR']]
print(show.to_string(index=False))
print("\nDIR = cases/100rb. RMSE_log/MAE_log = model optimization scale")

In [ ]:
print("="*70)
print("Table STGNN-Full per-seed, Dual Scale")
print("="*70)
print(f"{'Seed':<8}{'RMSE_log':>10}{'MAE_log':>10}{'RMSE_DIR':>11}{'MAE_DIR':>10}")
print("-"*70)
rows_t8 = []
for seed in SEEDS:
    d = all_seed_results[seed]['STGNN-Full']['test']
    m = dual_metrics(d['target'], d['pred'])
    rows_t8.append({'Seed': seed, **m})
    print(f"{seed:<8}{m['RMSE_log']:>10.4f}{m['MAE_log']:>10.4f}"
          f"{m['RMSE_DIR']:>11.3f}{m['MAE_DIR']:>10.3f}")

_arr = lambda k: _np.array([r[k] for r in rows_t8])
print("-"*70)
print(f"{'Mean±SD':<8}"
      f"{_arr('RMSE_log').mean():>6.4f}±{_arr('RMSE_log').std():.4f}"
      f" {_arr('MAE_log').mean():>5.4f}±{_arr('MAE_log').std():.4f}"
      f" {_arr('RMSE_DIR').mean():>5.2f}±{_arr('RMSE_DIR').std():.2f}"
      f" {_arr('MAE_DIR').mean():>5.2f}±{_arr('MAE_DIR').std():.2f}")
import pandas as _pd
_pd.DataFrame(rows_t8).to_csv('outputs/table8_perseed_dual.csv', index=False)

In [ ]:
acc = None
for seed in SEEDS:
    d = all_seed_results[seed]['STGNN-Full']['test']
    dfn = dual_metrics_per_node(d['target'], d['pred'], NODE_LIST)
    num = dfn.drop(columns='Kab/Kota')
    acc = num if acc is None else acc + num
df_pernode_fullseed = pd.concat(
    [pd.DataFrame({'Kab/Kota': NODE_LIST}), (acc / len(SEEDS)).reset_index(drop=True)],
    axis=1
).sort_values('RMSE_DIR', ascending=False)
 
df_pernode_fullseed.to_csv(
    'outputs/metrics_per_kabupaten_dual_meanseed.csv', index=False
)
print("Per-kabupaten STGNN-Full (rata-rata 5 seed) — top 10 RMSE_DIR:")
print(df_pernode_fullseed.head(10).round(
    {'Mean_DIR_aktual':2,'RMSE_log':4,'MAE_log':4,'RMSE_DIR':2,'MAE_DIR':2}
).to_string(index=False))
print("\nDisimpan: metrics_per_kabupaten_dual_meanseed.csv (untuk lampiran).")

In [ ]:
from scipy.stats import wilcoxon, rankdata
from statsmodels.stats.multitest import multipletests

OUT_DIR   = 'outputs'
SEEDS     = [42, 123, 456, 789, 2024]
BASELINES = ['CatBoost', 'SVM', 'CatBoost_lag12', 'SVM_lag12', 'LSTM', 'Ensemble_RF']
METRICS   = ['RMSE', 'MAE']

base_raw  = pd.read_csv(f'{OUT_DIR}/baseline_per_district_raw.csv')
stgnn_raw = pd.read_csv(f'{OUT_DIR}/stgnn_full_per_district_raw.csv')
raw       = pd.concat([base_raw, stgnn_raw], ignore_index=True)

def district_loss(df, metric):
    col = 'abs_err' if metric == 'MAE' else 'sq_err'
    l1  = df.groupby(['Model', 'Kab/Kota', 'month_idx'])[col].mean().reset_index()  # seed-avg
    l2  = l1.groupby(['Model', 'Kab/Kota'])[col].mean()                             # month-avg
    if metric == 'RMSE':
        l2 = np.sqrt(l2)
    return l2

results, per_district_out = [], {}
for metric in METRICS:
    wide = district_loss(raw, metric).unstack('Model').dropna()   # index=Kab/Kota
    per_district_out[metric] = wide
    if wide.shape[0] != 119:
        print(f'WARNING [{metric}]: n_districts = {wide.shape[0]} (harusnya 119). '
              f'Cek keselarasan nama Kab/Kota antar dua notebook.')
    stg = wide['STGNN-Full'].values
    for base_model in BASELINES:
        b    = wide[base_model].values
        diff = stg - b                          # <0 → STGNN lebih baik
        stat, p = wilcoxon(stg, b, alternative='less', zero_method='wilcox')
        nz  = diff[diff != 0]
        r   = rankdata(np.abs(nz))
        Wp, Wm = r[nz > 0].sum(), r[nz < 0].sum()
        rb  = (Wp - Wm) / (Wp + Wm) if (Wp + Wm) > 0 else 0.0   # rank-biserial (<0 = STGNN unggul)
        results.append({
            'Comparison'    : f'STGNN-Full vs {base_model}',
            'Metric'        : metric,
            'n_pairs'       : len(stg),
            'n_stgnn_better': int((diff < 0).sum()),
            'median_STGNN'  : float(np.median(stg)),
            'median_base'   : float(np.median(b)),
            'median_diff'   : float(np.median(diff)),
            'W'             : float(stat),
            'p_raw'         : float(p),
            'rank_biserial' : float(rb),
        })

res_df = pd.DataFrame(results)

reject, p_holm, _, _ = multipletests(res_df['p_raw'].values, method='holm')
res_df['p_holm']   = p_holm
res_df['sig_0.05'] = np.where(res_df['p_raw'] < 0.05, '*', 'ns')
res_df['sig_holm'] = np.where(reject, 'sig', 'ns')

res_df.to_csv(f'{OUT_DIR}/table_stat_validation_district_n119.csv', index=False)
for metric in METRICS:
    per_district_out[metric].to_csv(f'{OUT_DIR}/per_district_loss_{metric}_seedavg.csv')

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)
print('=' * 110)
print('PRIMARY STATISTICAL VALIDATION — Wilcoxon signed-rank, district-level (n = 119)')
print('=' * 110)
print(res_df.round(6).to_string(index=False))
print('\nSaved: table_stat_validation_district_n119.csv')
print('       per_district_loss_RMSE_seedavg.csv, per_district_loss_MAE_seedavg.csv')

In [ ]:
print("="*70)
print("TABLE STGNN-Full per-seed, dual scale")
print("="*70)
print(f"{'Seed':<8}{'RMSE_log':>10}{'MAE_log':>10}{'RMSE_DIR':>11}{'MAE_DIR':>10}")
print("-"*70)
rows_t8 = []
for seed in SEEDS:
    d = all_seed_results[seed]['STGNN-Full']['test']
    m = dual_metrics(d['target'], d['pred'])
    rows_t8.append({'Seed': seed, **m})
    print(f"{seed:<8}{m['RMSE_log']:>10.4f}{m['MAE_log']:>10.4f}"
          f"{m['RMSE_DIR']:>11.3f}{m['MAE_DIR']:>10.3f}")
 
_arr = lambda k: _np.array([r[k] for r in rows_t8])
print("-"*70)
print(f"{'Mean±SD':<8}"
      f"{_arr('RMSE_log').mean():>6.4f}±{_arr('RMSE_log').std():.4f}"
      f" {_arr('MAE_log').mean():>5.4f}±{_arr('MAE_log').std():.4f}"
      f" {_arr('RMSE_DIR').mean():>5.2f}±{_arr('RMSE_DIR').std():.2f}"
      f" {_arr('MAE_DIR').mean():>5.2f}±{_arr('MAE_DIR').std():.2f}")
import pandas as _pd
_pd.DataFrame(rows_t8).to_csv('outputs/table8_perseed_dual.csv', index=False)

#### Visualisasi Temporal Seluruh Periode

In [ ]:
import matplotlib.dates as mdates
# Predict full period
scope_mask = timestamps_pd < '2024-01-01'

# Filter timestamps dan X_raw
timestamps_scope    = [ts for ts, m in zip(timestamps, scope_mask) if m]
timestamps_pd_scope = pd.to_datetime(timestamps_scope)
X_raw_scope         = X_raw[scope_mask]

T_scope = X_raw_scope.shape[0]
print(f'Full X_raw shape       : {X_raw.shape}')
print(f'Scope X_raw shape      : {X_raw_scope.shape}')
print(f'Scope time range       : {timestamps_pd_scope.min()} → {timestamps_pd_scope.max()}')

X_full_sc = scaler.transform(
    X_raw_scope.reshape(-1, N_FEATURES)
).reshape(T_scope, N_NODES, N_FEATURES)
y_full_sc = X_full_sc[:, :, TARGET_IDX]

class FullPeriodDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, window):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.window = window
    
    def __len__(self):
        return len(self.X) - self.window
    
    def __getitem__(self, idx):
        return self.X[idx:idx+self.window], self.y[idx+self.window]


full_ds = FullPeriodDataset(X_full_sc, y_full_sc, WINDOW)
full_dl = torch.utils.data.DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=False)

model = STGNN(
    n_features=N_FEATURES, n_nodes=N_NODES, gcn_hidden=GCN_HIDDEN,
    tcn_hidden=TCN_HIDDEN, mlp_hidden=MLP_HIDDEN, kernel_size=KERNEL_SIZE,
    dropout=DROPOUT, node_embed_dim=NODE_EMBED,
).to(DEVICE)
model.load_state_dict(torch.load(
    'outputs/stgnn_standalone_seed42.pt',
    map_location=DEVICE
))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for x_batch, y_batch in full_dl:
        x_batch = x_batch.to(DEVICE)
        pred = model(x_batch, A_tensor)
        all_preds.append(pred.cpu().numpy())
        all_targets.append(y_batch.numpy())

pred_full_sc = np.concatenate(all_preds, axis=0)
tgt_full_sc  = np.concatenate(all_targets, axis=0)

mean0 = scaler.mean_[TARGET_IDX]
std0  = scaler.scale_[TARGET_IDX]
pred_full = pred_full_sc * std0 + mean0
tgt_full  = tgt_full_sc * std0 + mean0

full_timestamps = timestamps_pd_scope[WINDOW:]

print(f'\npred_full shape       : {pred_full.shape}')
print(f'tgt_full shape        : {tgt_full.shape}')
print(f'full_timestamps shape : ({len(full_timestamps)},)')
print(f'Prediction time range : {full_timestamps.min()} → {full_timestamps.max()}')

assert pred_full.shape[0] == len(full_timestamps), \
    f'Mismatch: pred {pred_full.shape[0]} vs ts {len(full_timestamps)}'
print('Shape match — siap plot')

In [ ]:
def plot_full_period_with_test_highlight(
    pred_full, tgt_full, timestamps_full, node_list,
    cities=None, n_cities=5,
    test_start='2023-01-01',
    save_path=None,
    use_dir_scale=False  # True = inverse log, False = DIR_log
):
    """
    Plot Actual vs Predicted untuk SELURUH periode,
    dengan area test set di-highlight.
    
    pred_full  : (T_full - window, N) — prediksi STGNN seluruh periode
    tgt_full   : (T_full - window, N) — actual seluruh periode
    """
    if cities is None:
        var_per_node = tgt_full.var(axis=0)
        top_idx = np.argsort(var_per_node)[::-1][:n_cities]
        cities = [node_list[i] for i in top_idx]
    
    cities = [c for c in cities if c in node_list]
    n_cities_actual = len(cities)
    
    test_start_dt = pd.Timestamp(test_start)
    
    fig, axes = plt.subplots(n_cities_actual, 1,
                             figsize=(14, 3.5 * n_cities_actual),
                             squeeze=False)
    
    for idx, city in enumerate(cities):
        ax  = axes[idx][0]
        n_i = node_list.index(city)
        
        if use_dir_scale:
            actual_vals = np.exp(tgt_full[:, n_i]) - 1
            pred_vals   = np.exp(pred_full[:, n_i]) - 1
            ylabel = 'DIR'
        else:
            actual_vals = tgt_full[:, n_i]
            pred_vals   = pred_full[:, n_i]
            ylabel = 'DIR_log'
        
        # Plot lines
        ax.plot(timestamps_full, actual_vals,
                label='Actual', color='steelblue', linewidth=1.5, alpha=0.9)
        ax.plot(timestamps_full, pred_vals,
                label='STGNN Ouput', color='tab:orange',
                linestyle='--', linewidth=1.3, alpha=0.85)

        # Highlight test period
        ax.axvspan(pd.Timestamp('2022-01-01'), timestamps_full.max(),
                   color='steelblue', alpha=0.12)
        ax.axvline(pd.Timestamp('2022-01-01'), color='steelblue',
                   linestyle='--', linewidth=1.2, alpha=0.7)
        
        test_mask_local = timestamps_full >= test_start_dt
        if test_mask_local.sum() > 0:
            test_rmse = np.sqrt(np.mean(
                (actual_vals[test_mask_local] - pred_vals[test_mask_local])**2
            ))
            ax.text(0.99, 0.96,
                    f'Test RMSE: {test_rmse:.3f}',
                    transform=ax.transAxes,
                    fontsize=9, verticalalignment='top',
                    horizontalalignment='right',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='white', alpha=0.85,
                              edgecolor='gray'))
        
        ax.set_title(city, fontsize=11, fontweight='bold')
        ax.set_xlabel('Year')
        ax.set_ylabel(ylabel)
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)
        
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.tick_params(axis='x', rotation=0)
    
    plt.suptitle('Plot Actual vs STGNN Output (2005-2023)',
                 fontsize=13, fontweight='bold', y=1.005)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Plot disimpan: {save_path}')
    plt.show()

In [ ]:
# Generate plot
sample_cities = [
    'KOTA BANDUNG',
    'JOMBANG',
    'KOTA SEMARANG',
    'KOTA JAKARTA SELATAN',
    'NGANJUK'
]

# Plot dengan skala DIR_log
plot_full_period_with_test_highlight(
    pred_full, tgt_full, full_timestamps, NODE_LIST,
    cities=sample_cities,
    test_start='2023-01-01',
    save_path='outputs/actual_vs_predicted_full.png',
    use_dir_scale=False
)

plot_full_period_with_test_highlight(
    pred_full, tgt_full, full_timestamps, NODE_LIST,
    cities=sample_cities,
    test_start='2023-01-01',
    save_path='outputs/actual_vs_predicted_full_DIR.png',
    use_dir_scale=True
)

#### Spatial Mapping DIR

In [ ]:
import json
import urllib.request
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.patches import PathPatch, Patch
from matplotlib.path import Path

# Source: Humanitarian Data Exchange (HDX)
GEOJSON_URL   = 'https://raw.githubusercontent.com/JfrAziz/indonesia-district/master/kab%2034.geojson'
GEOJSON_LOCAL = 'data/indonesia_kabupaten.geojson'
 
if not os.path.exists(GEOJSON_LOCAL):
    print('Downloading geojson dari HDX (sekali saja)...')
    os.makedirs(os.path.dirname(GEOJSON_LOCAL), exist_ok=True)
    urllib.request.urlretrieve(GEOJSON_URL, GEOJSON_LOCAL)
    print(f'Tersimpan di: {GEOJSON_LOCAL}')
 
with open(GEOJSON_LOCAL) as f:
    gj_all = json.load(f)

In [ ]:
java_prov_ids = {'31', '32', '33', '34', '35', '36'}
java_features = [f for f in gj_all['features']
                 if f['properties']['prov_id'] in java_prov_ids]
geom_lookup   = {f['properties']['name']: f['geometry'] for f in java_features}
print(f'Jawa features dari geojson: {len(java_features)}')

In [ ]:
missing = [n for n in NODE_LIST if n not in geom_lookup]
if missing:
    raise ValueError(f'Kabupaten berikut tidak ada di geojson: {missing}')
print(f'Semua {len(NODE_LIST)} kabupaten match dengan geojson')
 

In [ ]:
df_centroid = pd.read_csv('data/df_centroid.csv')
df_geo = df_centroid.set_index('Kab/Kota').loc[NODE_LIST].reset_index()
lons = df_geo['longitude'].values
lats = df_geo['latitude'].values

In [ ]:
def _geom_to_path(geom):
    verts, codes = [], []
    polys = geom['coordinates'] if geom['type'] == 'MultiPolygon' else [geom['coordinates']]
    for poly in polys:
        for ring in poly:
            if len(ring) < 3:
                continue
            r = np.array(ring)
            verts.extend(r.tolist())
            codes.extend([Path.MOVETO]
                         + [Path.LINETO] * (len(r) - 2)
                         + [Path.CLOSEPOLY])
    return Path(verts, codes)
 
paths = [_geom_to_path(geom_lookup[n]) for n in NODE_LIST]

In [ ]:
BBOX      = [105.0, 115.0, -9.0, -5.5]
SEA_COLOR = '#eaf4fb'                     
OUT_DIR   = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
 
def _draw_choropleth(ax, values, cmap, norm, ec='white', lw=0.4):
    for i, path in enumerate(paths):
        ax.add_patch(PathPatch(path, facecolor=cmap(norm(values[i])),
                               edgecolor=ec, linewidth=lw))
 
def _setup_axes(ax, title):
    ax.set_xlim(BBOX[0], BBOX[1]); ax.set_ylim(BBOX[2], BBOX[3])
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(title, fontsize=11.5, fontweight='bold')
    ax.grid(True, alpha=0.25); ax.set_aspect('equal'); ax.set_facecolor(SEA_COLOR)

In [ ]:
CANONICAL_SEED = 42

data = np.load(
    f'outputs/predictions_full_seed{CANONICAL_SEED}.npz'
)
pred_test_orig   = data['pred_test_orig']
target_test_orig = data['target_test_orig']

print(f'Loaded predictions from seed={CANONICAL_SEED}')
print(f'  pred_test_orig shape  : {pred_test_orig.shape}')
print(f'  target_test_orig shape: {target_test_orig.shape}')

In [ ]:
rmse_per_node = np.sqrt(np.mean((target_test_orig - pred_test_orig) ** 2, axis=0))
mae_per_node = np.mean(np.abs(target_test_orig - pred_test_orig), axis=0)

df_geo['Test_RMSE'] = rmse_per_node
df_geo['Test_MAE'] = mae_per_node

target_test_dir = np.expm1(target_test_orig)
pred_test_dir   = np.expm1(pred_test_orig)
rmse_per_node_dir = np.sqrt(np.mean((target_test_dir - pred_test_dir) ** 2, axis=0))
mae_per_node_dir  = np.mean(np.abs(target_test_dir - pred_test_dir), axis=0)
df_geo['Test_RMSE_DIR'] = rmse_per_node_dir
df_geo['Test_MAE_DIR']  = mae_per_node_dir

fig, ax = plt.subplots(figsize=(15, 7))
norm = Normalize(vmin=np.percentile(rmse_per_node, 5),
                 vmax=np.percentile(rmse_per_node, 95))
_draw_choropleth(ax, rmse_per_node, plt.cm.YlOrRd, norm)
 
sm = plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd, norm=norm); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label('Test RMSE (DIR_log)', fontsize=11)

worst3 = df_geo.nlargest(3, 'Test_RMSE')
best3  = df_geo.nsmallest(3, 'Test_RMSE')
def _annot(row, color, dy):
    ax.annotate(f"{row['Kab/Kota']}\n{row['Test_RMSE']:.3f}",
                xy=(row['longitude'], row['latitude']),
                xytext=(0, dy), textcoords='offset points',
                ha='center', fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.25', fc='white', ec=color, lw=0.8, alpha=0.92),
                arrowprops=dict(arrowstyle='-', color=color, lw=0.6))
for _, r in worst3.iterrows(): _annot(r, 'darkred', 18)
for _, r in best3.iterrows():  _annot(r, 'darkgreen', -28)
 
ax.set_xlim(BBOX[0], BBOX[1]); ax.set_ylim(BBOX[2], BBOX[3])
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(
    f'Spasial Mapping Test RMSE per Kabupaten/Kota - STGNN\n'
    f'Mean={rmse_per_node.mean():.3f} | Median={np.median(rmse_per_node):.3f} | '
    f'Min={rmse_per_node.min():.3f} | Max={rmse_per_node.max():.3f}',
    fontsize=11.5)
ax.grid(True, alpha=0.25); ax.set_aspect('equal'); ax.set_facecolor(SEA_COLOR)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/map_rmse_choropleth.png', dpi=140, bbox_inches='tight')
plt.show()

df_geo[['Provinsi','Kab/Kota','latitude','longitude','Test_RMSE', 'Test_MAE','Test_RMSE_DIR','Test_MAE_DIR']] \
    .sort_values('Test_RMSE') \
    .to_csv(f'{OUT_DIR}/rmse_per_kabkota_baru.csv', index=False)
 
print('\nTop 5 with the smallest RMSE:')
print(df_geo.nsmallest(5,'Test_RMSE')[['Kab/Kota','Test_RMSE']].to_string(index=False))
print('\nTop 5 models with the highest RMSE:')
print(df_geo.nlargest(5,'Test_RMSE')[['Kab/Kota','Test_RMSE']].to_string(index=False))

In [ ]:
results = df_geo[['Provinsi', 'Kab/Kota',
                  'Test_RMSE', 'Test_MAE']].copy()

results = results.sort_values('Kab/Kota')

results.to_csv(f'{OUT_DIR}/rmse_mae_per_kabkota.csv', index=False)

print("Saved:", f'{OUT_DIR}/rmse_mae_per_kabkota.csv')

In [ ]:
mean_pred   = pred_test_orig.mean(axis=0)
mean_actual = target_test_orig.mean(axis=0)
q33f, q66f = np.percentile(mean_actual, [33.33, 66.67])
 
risk_a = _classify(mean_actual, q33f, q66f)
risk_p = _classify(mean_pred,   q33f, q66f)
agree_full = (risk_a == risk_p).mean() * 100
 
t0 = pd.to_datetime(test_timestamps[0]).strftime('%b %Y')
t1 = pd.to_datetime(test_timestamps[-1]).strftime('%b %Y')
 
fig, axes = plt.subplots(1, 2, figsize=(19, 7))
for ax, risk_arr, title in [
    (axes[0], risk_a, f'AKTUAL - Rata-rata {t0} s.d. {t1}'),
    (axes[1], risk_p, f'PREDIKSI STGNN - Rata-rata {t0} s.d. {t1}'),
]:
    _draw_choropleth(ax, risk_arr, risk_cmap, risk_norm)
    counts = [(risk_arr == k).sum() for k in range(3)]
    handles = [Patch(facecolor=risk_cmap.colors[k], edgecolor='black',
                     label=f'{risk_labels[k]} (n={counts[k]})') for k in range(3)]
    ax.legend(handles=handles, title='Kategori Risiko',
              loc='lower right', fontsize=9, title_fontsize=10)
    _setup_axes(ax, title)
 
plt.suptitle(
    f'Pemetaan Risiko DBD Rata-rata Periode Test\n'
    f'Risk-class Agreement: {agree_full:.1f}%',
    fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/map_risk_mean_choropleth.png', dpi=140, bbox_inches='tight')
plt.show()

df_risk_full = pd.DataFrame({
    'Provinsi'        : df_geo['Provinsi'].values,
    'Kab/Kota'        : NODE_LIST,
    'latitude'        : lats,
    'longitude'       : lons,
    'DIR_actual_mean' : mean_actual,
    'DIR_pred_mean'   : mean_pred,
    'Risk_Actual'     : [risk_labels[k].split()[0] for k in risk_a],
    'Risk_Predicted'  : [risk_labels[k].split()[0] for k in risk_p],
    'Match'           : (risk_a == risk_p),
})
df_risk_full.to_csv(f'{OUT_DIR}/risk_classification_full.csv', index=False)
print(f'Mean-period risk agreement: {agree_full:.1f}%')
print(f'Mismatch: {(~df_risk_full["Match"]).sum()} dari {len(NODE_LIST)} kabupaten')


In [ ]:
# MAP: Actual vs Prediction Risk Class 
from sklearn.metrics import cohen_kappa_score, confusion_matrix

mean_pred   = pred_test_orig.mean(axis=0)
mean_actual = target_test_orig.mean(axis=0)

# Threshold dari aktual saja → ground truth sebagai reference
q33f, q66f = np.percentile(mean_actual, [33.33, 66.67])
risk_a = _classify(mean_actual, q33f, q66f)
risk_p = _classify(mean_pred,   q33f, q66f)


# Metrik agreement
agree_full      = (risk_a == risk_p).mean() * 100
kappa_quadratic = cohen_kappa_score(risk_a, risk_p, weights='quadratic')
kappa_linear    = cohen_kappa_score(risk_a, risk_p, weights='linear')
kappa_unweight  = cohen_kappa_score(risk_a, risk_p)

# Confusion matrix untuk analisis boundary cases
cm = confusion_matrix(risk_a, risk_p, labels=[0, 1, 2])
boundary_jump = cm[0, 2] + cm[2, 0]
adjacent_err  = cm[0, 1] + cm[1, 0] + cm[1, 2] + cm[2, 1]

def _kappa_interpret(k):
    if k < 0.00:  return "Poor"
    if k < 0.21:  return "Slight"
    if k < 0.41:  return "Fair"
    if k < 0.61:  return "Moderate"
    if k < 0.81:  return "Substantial"
    return "Almost Perfect"

print("=" * 60)
print(f"AGREEMENT METRICS — Seed 42 (canonical)")
print("=" * 60)
print(f"Percentage Agreement      : {agree_full:.2f}%")
print(f"Cohen's Kappa (quadratic) : {kappa_quadratic:.4f}  → {_kappa_interpret(kappa_quadratic)}")
print(f"Cohen's Kappa (linear)    : {kappa_linear:.4f}  → {_kappa_interpret(kappa_linear)}")
print(f"Cohen's Kappa (unweight)  : {kappa_unweight:.4f}  → {_kappa_interpret(kappa_unweight)}")
print(f"Misclassified             : {(~(risk_a == risk_p)).sum()} dari {len(risk_a)}")
print(f"  - Adjacent (1 kelas)    : {adjacent_err}")
print(f"  - Boundary (Low↔High)   : {boundary_jump}")
print("-" * 60)
print("Confusion Matrix:")
print("            Pred_Low  Pred_Mod  Pred_High")
print(f"Actual_Low    {cm[0,0]:4d}     {cm[0,1]:4d}     {cm[0,2]:4d}")
print(f"Actual_Mod    {cm[1,0]:4d}     {cm[1,1]:4d}     {cm[1,2]:4d}")
print(f"Actual_High   {cm[2,0]:4d}     {cm[2,1]:4d}     {cm[2,2]:4d}")
print("=" * 60)

t0 = pd.to_datetime(test_timestamps[0]).strftime('%b %Y')
t1 = pd.to_datetime(test_timestamps[-1]).strftime('%b %Y')

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(19, 7))
for ax, risk_arr, title in [
    (axes[0], risk_a, f'AKTUAL - Rata-rata {t0} s.d. {t1}'),
    (axes[1], risk_p, f'PREDIKSI STGNN - Rata-rata {t0} s.d. {t1}'),
]:
    _draw_choropleth(ax, risk_arr, risk_cmap, risk_norm)
    counts = [(risk_arr == k).sum() for k in range(3)]
    handles = [Patch(facecolor=risk_cmap.colors[k], edgecolor='black',
                     label=f'{risk_labels[k]} (n={counts[k]})') for k in range(3)]
    ax.legend(handles=handles, title='Kategori Risiko',
              loc='lower right', fontsize=9, title_fontsize=10)
    _setup_axes(ax, title)

plt.suptitle(
    f'Pemetaan Risiko DBD Rata-rata Periode Test\n'
    f'Risk-class Agreement: {agree_full:.1f}%  ·  '
    f"Weighted Kappa: {kappa_quadratic:.3f} ({_kappa_interpret(kappa_quadratic)})",
    fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/map_risk_mean_choropleth.png', dpi=140, bbox_inches='tight')
plt.show()

# Save tabel & summary
df_risk_full = pd.DataFrame({
    'Provinsi'        : df_geo['Provinsi'].values,
    'Kab/Kota'        : NODE_LIST,
    'latitude'        : lats,
    'longitude'       : lons,
    'DIR_actual_mean' : mean_actual,
    'DIR_pred_mean'   : mean_pred,
    'Risk_Actual'     : [risk_labels[k].split()[0] for k in risk_a],
    'Risk_Predicted'  : [risk_labels[k].split()[0] for k in risk_p],
    'Match'           : (risk_a == risk_p),
})
df_risk_full.to_csv(f'{OUT_DIR}/risk_classification_full.csv', index=False)
print(f'\nMean-period risk agreement: {agree_full:.1f}%')
print(f'Weighted Kappa: {kappa_quadratic:.3f}')
print(f'Mismatch: {(~df_risk_full["Match"]).sum()} dari {len(NODE_LIST)} kabupaten')

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(risk_a, risk_p, labels=[0, 1, 2])

print("Confusion Matrix (rows=actual, cols=prediction):")
print(f"              Pred_Low  Pred_Mod  Pred_High")
print(f"Actual_Low      {cm[0,0]:4d}     {cm[0,1]:4d}     {cm[0,2]:4d}    | total: {cm[0].sum()}")
print(f"Actual_Mod      {cm[1,0]:4d}     {cm[1,1]:4d}     {cm[1,2]:4d}    | total: {cm[1].sum()}")
print(f"Actual_High     {cm[2,0]:4d}     {cm[2,1]:4d}     {cm[2,2]:4d}    | total: {cm[2].sum()}")
print(f"-" * 60)
print(f"Pred totals     {cm[:,0].sum():4d}     {cm[:,1].sum():4d}     {cm[:,2].sum():4d}")

print(f"\nDiagonal (correct): {np.trace(cm)} / {cm.sum()}")
print(f"Off-diagonal (misclassified): {cm.sum() - np.trace(cm)}")

#### Java Island Map - 119 Nodes + Edges

In [ ]:
OUT_DIR = 'outputs'
 
fig, ax = plt.subplots(figsize=(16, 8))

patches = [PathPatch(p, facecolor='#f5f5f5', edgecolor='#cccccc', linewidth=0.4) 
           for p in paths]
ax.add_collection(PatchCollection(patches, match_original=True))
 
edge_segments = []
edge_weights  = []
for i in range(N_NODES):
    for j in range(i+1, N_NODES):
        if A_hybrid[i, j] > 0:
            edge_segments.append([(lons[i], lats[i]), (lons[j], lats[j])])
            edge_weights.append(A_hybrid[i, j])
 
edge_weights = np.array(edge_weights)
print(f'Total edges (undirected): {len(edge_segments)}')

norm = Normalize(vmin=edge_weights.min(), vmax=edge_weights.max())
cmap = plt.get_cmap('plasma')
edge_colors = cmap(norm(edge_weights))
edge_colors[:, 3] = 0.15 + 0.7 * norm(edge_weights)
 
lc = LineCollection(edge_segments, colors=edge_colors, 
                    linewidths=0.3 + 1.5 * norm(edge_weights))
ax.add_collection(lc)

degrees = A_hybrid.astype(bool).sum(axis=1)
ax.scatter(lons, lats, s=20 + degrees * 2, c='#d62728', 
           edgecolor='white', linewidth=0.5, zorder=5, alpha=0.85)
 
ax.set_xlim(105.0, 115.0)
ax.set_ylim(-9.0, -5.5)
ax.set_facecolor('#eaf4fb')
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title(
    f'Spatial Graph Representation - Java Island\n'
    f'{N_NODES} Nodes (Regencies/Cities) and {len(edge_segments)} Edges',
    fontsize=12, fontweight='bold'
)
ax.set_aspect('equal')

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.7, aspect=20, pad=0.02)
cbar.set_label('Bobot edge (Gaussian weight)', fontsize=10)
 
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_graph_java_overview.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved → fig_graph_java_overview.png\n')

#### Graph Constructions - Kota Bandung + Nearest Neighbors

In [ ]:
TARGET_KAB = 'KOTA BANDUNG'
 
if TARGET_KAB not in NODE_LIST:
    candidates = [n for n in NODE_LIST if 'BANDUNG' in n.upper()]
    print(f'⚠ "{TARGET_KAB}" tidak ditemukan. Kandidat: {candidates}')
    TARGET_KAB = candidates[0] if candidates else NODE_LIST[0]
    print(f'  Menggunakan: {TARGET_KAB}\n')
 
idx_target = NODE_LIST.index(TARGET_KAB)

distances_from_target = D_km[idx_target].copy()
distances_from_target[idx_target] = np.inf  
neighbor_indices = np.argsort(distances_from_target)[:K]
 
neighbor_info = []
for j in neighbor_indices:
    d_km   = D_km[idx_target, j]
    weight = float(np.exp(-(d_km ** 2) / (2 * SIGMA ** 2)))
    neighbor_info.append({
        'name'    : NODE_LIST[j],
        'idx'     : j,
        'lon'     : lons[j],
        'lat'     : lats[j],
        'distance': d_km,
        'weight'  : weight,
    })
 
neighbor_info.sort(key=lambda x: x['distance'])
 
print(f'10 tetangga terdekat dari {TARGET_KAB}:')
print(f'{"No":<3} {"Kabupaten/Kota":<25} {"Jarak (km)":>11} {"Weight":>9}')
print('-' * 52)
for i, n in enumerate(neighbor_info, 1):
    print(f'{i:<3} {n["name"]:<25} {n["distance"]:>11.2f} {n["weight"]:>9.4f}')
 
# Build figure 
fig = plt.figure(figsize=(17, 7))
gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 1], wspace=0.25)
ax_map = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1])
 
all_lons = [lons[idx_target]] + [n['lon'] for n in neighbor_info]
all_lats = [lats[idx_target]] + [n['lat'] for n in neighbor_info]
pad = 0.3
xmin, xmax = min(all_lons) - pad, max(all_lons) + pad
ymin, ymax = min(all_lats) - pad, max(all_lats) + pad
 
view_patches = []
target_patch = None
neighbor_patches_local = []
 
for i, p in enumerate(paths):
    name_i = NODE_LIST[i]
    if i == idx_target:
        target_patch = PathPatch(p, facecolor='#ffe5e5', edgecolor='#c0392b',
                                  linewidth=1.8, zorder=3)
        ax_map.add_patch(target_patch)
    elif i in neighbor_indices:
        np_ = PathPatch(p, facecolor='#e8f3ff', edgecolor='#2980b9',
                        linewidth=1.0, zorder=2)
        ax_map.add_patch(np_)
        neighbor_patches_local.append(np_)
    else:
        bg = PathPatch(p, facecolor='#f8f8f8', edgecolor='#dddddd',
                       linewidth=0.3, zorder=1)
        ax_map.add_patch(bg)
 
weights_arr = np.array([n['weight'] for n in neighbor_info])
norm_e = Normalize(vmin=weights_arr.min(), vmax=weights_arr.max())
cmap_e = plt.get_cmap('plasma')
 
for n in neighbor_info:
    color = cmap_e(norm_e(n['weight']))
    width = 0.8 + 4.0 * norm_e(n['weight'])
    ax_map.plot(
        [lons[idx_target], n['lon']],
        [lats[idx_target], n['lat']],
        color=color, linewidth=width, alpha=0.85, zorder=4,
        solid_capstyle='round',
    )
 
# Node markers
ax_map.scatter(lons[idx_target], lats[idx_target], s=280, c='#c0392b',
               edgecolor='white', linewidth=2, zorder=6, marker='*',
               label=f'{TARGET_KAB} (target)')
for n in neighbor_info:
    ax_map.scatter(n['lon'], n['lat'], s=80, c='#2980b9',
                   edgecolor='white', linewidth=1.2, zorder=5)
    label = n['name'].replace('KABUPATEN ', 'Kab. ').replace('KOTA ', 'Kota ').title()
    ax_map.annotate(label, (n['lon'], n['lat']),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=8, color='#2c3e50', alpha=0.9,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                              edgecolor='none', alpha=0.7))
 
ax_map.set_xlim(xmin, xmax)
ax_map.set_ylim(ymin, ymax)
ax_map.set_facecolor('#eaf4fb')
ax_map.set_xlabel('Longitude', fontsize=10)
ax_map.set_ylabel('Latitude', fontsize=10)
ax_map.set_title(
    f'Graf Spasial {TARGET_KAB.title()} dan {K} Tetangga Terdekat (KNN)\n',
    fontsize=11, fontweight='bold'
)
ax_map.set_aspect('equal')
ax_map.legend(loc='upper right', fontsize=9, framealpha=0.95)

sm_e = ScalarMappable(norm=norm_e, cmap=cmap_e)
sm_e.set_array([])
cbar_e = plt.colorbar(sm_e, ax=ax_map, shrink=0.5, pad=0.02)
cbar_e.set_label('Bobot Gaussian', fontsize=9)

 
names_short = [n['name'].replace('KABUPATEN ', '').replace('KOTA ', '').title()
               for n in neighbor_info]
distances   = [n['distance'] for n in neighbor_info]
weights_v   = [n['weight']   for n in neighbor_info]
 
y_pos = np.arange(len(neighbor_info))
 
# Bar chart: distance (left axis) + weight overlay (right axis)
bars = ax_bar.barh(y_pos, distances, color=cmap_e(norm_e(np.array(weights_v))),
                    edgecolor='#2c3e50', linewidth=0.6, alpha=0.9)
 
for i, (bar, w, d) in enumerate(zip(bars, weights_v, distances)):
    ax_bar.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                f'd={d:.1f}km\nw={w:.3f}',
                va='center', fontsize=8.5, color='#2c3e50')
 
ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(names_short, fontsize=9)
ax_bar.invert_yaxis()  # closest at top
ax_bar.set_xlabel('Jarak Haversine (km)', fontsize=10)
ax_bar.set_ylabel('Kab/Kota', fontsize=10)
ax_bar.set_xlim(0, max(distances) * 1.35)
ax_bar.set_title(
    f'Jarak dan Bobot Gaussian per Tetangga\n',
    fontsize=11, fontweight='bold'
)
ax_bar.grid(axis='x', alpha=0.3)
ax_bar.set_axisbelow(True)
 
plt.suptitle(
    f'Ilustrasi Konstruksi Graf Spasial {TARGET_KAB.title()}',
    fontsize=13, fontweight='bold', y=1.03
)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_graph_bandung_illustration.png',
            dpi=200, bbox_inches='tight')
plt.show()
print(f'\nSaved → fig_graph_bandung_illustration.png')
 
df_neighbors = pd.DataFrame(neighbor_info)
df_neighbors['rank'] = range(1, len(df_neighbors) + 1)
df_neighbors = df_neighbors[['rank', 'name', 'distance', 'weight']]
df_neighbors.columns = ['Rank', 'Kabupaten/Kota', 'Jarak (km)', 'Bobot Gaussian']
df_neighbors.to_csv(f'{OUT_DIR}/neighbors_bandung.csv', index=False)
print(f'Saved → neighbors_bandung.csv')